# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa

This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We'll interact with the dataset via its Croissant schema, referencing record sets, fields, and columns by their `@id`s.

### Dataset Source

The dataset Croissant schema is available at:
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading

Let's use `mlcroissant` to load the Croissant metadata and list general information about the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", getattr(metadata, 'datePublished', 'N/A'))
print("Cite as:", getattr(metadata, 'citeAs', 'N/A'))

## 2. Data Overview
Let's inspect the available **record sets** and their schema. We'll enumerate the dataset's record sets, referencing each by its `@id`, and summarize fields and columns for the main record set.

In [ ]:
# List all record sets by their @id
all_record_sets = dataset.record_sets()
print(f"Number of Record Sets: {len(all_record_sets)}\n")
for recset in all_record_sets:
    print(f"RecordSet ID: {recset.id}")
    print(f"  Name: {recset.name}")
    if hasattr(recset, 'description'):
        print(f"  Description: {recset.description}")
    print(f"  Fields: {[field.id for field in recset.fields]}")
    # Show fields and their datatypes
    for field in recset.fields:
        print(f"    Field ID: {field.id}, Name: {field.name}, DataType: {getattr(field, 'data_type', 'N/A')}")
    print('-'*60)

**Let's peek at a few records from the main RecordSet.**

_(Replace the variable below with record set `@id` from the previous cell; for this dataset, the main table is likely identified as `cr:RecordSet` or similar.)_

In [ ]:
# Choose the main survey RecordSet (replace as needed based on overview above)
MAIN_RECORDSET_ID = 'cr:RecordSet_1'  # Adjust according to the printout above

try:
    for i, record in enumerate(dataset.records(record_set=MAIN_RECORDSET_ID)):
        print(record)
        if i >= 2:
            break
except Exception as e:
    print(f"Error fetching records with RecordSet {MAIN_RECORDSET_ID}. Please ensure the variable matches a listed RecordSet @id.")
    print(e)

## 3. Data Extraction

Let's load the main survey record set into a Pandas DataFrame, referencing by explicit `@id` so downstream processing is robust and readable. (You can repeat this for other RecordSets as needed.)

In [ ]:
# If there are multiple record sets, list all their IDs to load all
record_sets_ids = [recset.id for recset in dataset.record_sets()]
dataframes = {}
for recset_id in record_sets_ids:
    print(f"Loading records from RecordSet: {recset_id}")
    data = list(dataset.records(record_set=recset_id))
    if data:
        df = pd.DataFrame(data)
        dataframes[recset_id] = df
        print(f"  Loaded {len(df)} rows, columns: {df.columns.tolist()}")
    else:
        print("  No data in this RecordSet.")
print("Done loading all record sets.")

In [ ]:
# Preview the main DataFrame (change to correct RecordSet ID if needed)
main_df = dataframes.get(MAIN_RECORDSET_ID, None)
if main_df is not None:
    print(main_df.columns.tolist())
    main_df.head()
else:
    print(f"No DataFrame for RecordSet {MAIN_RECORDSET_ID}.")

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (for example: PHQ-9 score, GAD-7 score, age, or similar numeric measurement) for analysis. All fields and their IDs can be seen in the schema above. We will filter, normalize, and group the data, referencing fields by their `@id`.

In [ ]:
# Choose a numeric field and group field by their @id as seen in schema/overview
NUMERIC_FIELD_ID = 'phq9_total_score'   # Replace with true @id from your field list
GROUP_FIELD_ID = 'gender'              # Replace with @id (e.g., 'gender', 'village', etc.)

if main_df is not None and NUMERIC_FIELD_ID in main_df.columns:
    # Remove rows with missing values for the analysis
    df_numeric = main_df.dropna(subset=[NUMERIC_FIELD_ID])
    # Example threshold filtering
    threshold = 10
    filtered_df = df_numeric[df_numeric[NUMERIC_FIELD_ID] > threshold]
    print(f"Filtered records with {NUMERIC_FIELD_ID} > {threshold}: {filtered_df.shape[0]} records")
    print(filtered_df[[NUMERIC_FIELD_ID]].head())

    # Normalize the field
    norm_col = f"{NUMERIC_FIELD_ID}_normalized"
    filtered_df[norm_col] = (filtered_df[NUMERIC_FIELD_ID] - filtered_df[NUMERIC_FIELD_ID].mean()) / filtered_df[NUMERIC_FIELD_ID].std()
    print(filtered_df[[NUMERIC_FIELD_ID, norm_col]].head())

    # Group by a categorical field if present
    if GROUP_FIELD_ID in main_df.columns:
        grouped_df = filtered_df.groupby(GROUP_FIELD_ID)[NUMERIC_FIELD_ID].mean().reset_index()
        print(f"Mean {NUMERIC_FIELD_ID} grouped by {GROUP_FIELD_ID}:")
        print(grouped_df.head())
    else:
        print(f"Group field '{GROUP_FIELD_ID}' not found in columns: {main_df.columns.tolist()}")
else:
    print(f"The field {NUMERIC_FIELD_ID} was not found in this DataFrame's columns:")
    print(main_df.columns.tolist() if main_df is not None else "No DataFrame found.")

## 5. Visualization

Let's plot the distribution of the chosen numeric field and compare means by group. (You may need to adjust the field names to match actual column @id's.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and NUMERIC_FIELD_ID in main_df.columns:
    # Plot distribution
    plt.figure(figsize=(8, 6))
    sns.histplot(main_df[NUMERIC_FIELD_ID].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {NUMERIC_FIELD_ID}")
    plt.xlabel(NUMERIC_FIELD_ID)
    plt.ylabel("Frequency")
    plt.show()

    # Grouped boxplot
    if GROUP_FIELD_ID in main_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=main_df[GROUP_FIELD_ID], y=main_df[NUMERIC_FIELD_ID])
        plt.title(f"{NUMERIC_FIELD_ID} by {GROUP_FIELD_ID}")
        plt.xlabel(GROUP_FIELD_ID)
        plt.ylabel(NUMERIC_FIELD_ID)
        plt.show()
else:
    print(f"Cannot visualize: {NUMERIC_FIELD_ID} not found.")

## 6. Conclusion

- We loaded, inspected, and processed the FAIR² Mental Health Survey Dataset using `mlcroissant`.
- We referenced all schema and data elements by their Croissant `@id` for clarity and reproducibility.
- Exploratory steps demonstrated filtering, normalizing, and visualizing survey responses, giving insights into distributions and group differences for key mental health indicators.

Next steps could include more advanced modeling, data FAIRness assessment, or linking to downstream public health applications.

> **Tip:** When using your own analysis, always double-check the `@id` for each field, especially if the schema evolves or changes.